In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Set catalog and schema
catalog = "automobile_catalog"
schema = "003_gold"

print(f"Generating KPIs from: {catalog}.{schema}")

In [0]:
# MTD Performance vs Previous MTD
# Month-to-date revenue and completed orders compared with previous MTD (January 2026 vs December 2025)

from pyspark.sql import functions as F

# Read gold layer tables
fact_invoices = spark.table(f"{catalog}.{schema}.fact_invoices")
dim_store = spark.table(f"{catalog}.{schema}.dim_store")

# Define MTD periods (using latest available data: January 2026 vs December 2025)
current_mtd_start = '2026-01-01'
current_mtd_end = '2026-01-30'
previous_mtd_start = '2025-12-01'
previous_mtd_end = '2025-12-30'

# Filter completed orders and join with dim_store
completed_orders = (
    fact_invoices
    .filter(F.col("order_status") == "COMPLETED")
    .join(dim_store, "store_id", "inner")
)

# Calculate current MTD metrics
current_mtd = completed_orders.filter(
    (F.col("invoice_date") >= current_mtd_start) & 
    (F.col("invoice_date") <= current_mtd_end)
).groupBy(
    "store_id", "store_name", "manager_id", "manager_name"
).agg(
    F.sum("invoice_amount").alias("current_mtd_revenue"),
    F.count("order_id").alias("current_mtd_orders")
)

# Calculate previous MTD metrics
previous_mtd = completed_orders.filter(
    (F.col("invoice_date") >= previous_mtd_start) & 
    (F.col("invoice_date") <= previous_mtd_end)
).groupBy(
    "store_id", "store_name", "manager_id", "manager_name"
).agg(
    F.sum("invoice_amount").alias("previous_mtd_revenue"),
    F.count("order_id").alias("previous_mtd_orders")
)

# Combine current and previous MTD
mtd_comparison = current_mtd.join(
    previous_mtd,
    ["store_id", "store_name", "manager_id", "manager_name"],
    "full_outer"
).fillna(0, subset=["current_mtd_revenue", "current_mtd_orders", "previous_mtd_revenue", "previous_mtd_orders"])

# Calculate performance metrics
mtd_kpi = mtd_comparison.withColumn(
    "revenue_change",
    F.col("current_mtd_revenue") - F.col("previous_mtd_revenue")
).withColumn(
    "revenue_change_pct",
    F.when(F.col("previous_mtd_revenue") > 0,
           ((F.col("current_mtd_revenue") - F.col("previous_mtd_revenue")) / F.col("previous_mtd_revenue") * 100)
    ).otherwise(None)
).withColumn(
    "orders_change",
    F.col("current_mtd_orders") - F.col("previous_mtd_orders")
).withColumn(
    "orders_change_pct",
    F.when(F.col("previous_mtd_orders") > 0,
           ((F.col("current_mtd_orders") - F.col("previous_mtd_orders")) / F.col("previous_mtd_orders") * 100)
    ).otherwise(None)
).select(
    "store_id",
    "store_name",
    "manager_id",
    "manager_name",
    F.round("current_mtd_revenue", 2).alias("current_mtd_revenue"),
    F.col("current_mtd_orders"),
    F.round("previous_mtd_revenue", 2).alias("previous_mtd_revenue"),
    F.col("previous_mtd_orders"),
    F.round("revenue_change", 2).alias("revenue_change"),
    F.round("revenue_change_pct", 2).alias("revenue_change_pct"),
    F.col("orders_change"),
    F.round("orders_change_pct", 2).alias("orders_change_pct")
).orderBy(F.desc("current_mtd_revenue"))

print("\n📊 KPI 1: MTD Performance vs Previous MTD (Jan 2026 vs Dec 2025)")
print("="*80)
display(mtd_kpi)

In [0]:
from pyspark.sql import functions as F

operations = spark.table(f"{catalog}.{schema}.cube_operations_detail")

# Clean & validate data
operations_clean = (
    operations
    .filter(F.col("days_in_shop").isNotNull())
    .filter(F.col("days_in_shop") >= 0)         # remove negative values
    .filter(F.col("store_id").isNotNull())
    .filter(F.col("service_type").isNotNull())
)

avg_days_in_shop = (
    operations_clean
    .groupBy("store_id", "store_name", "service_type")
    .agg(
        F.round(F.avg("days_in_shop"), 2).alias("avg_days_in_shop"),
        F.count("*").alias("total_orders"),
        F.min("days_in_shop").alias("min_days"),
        F.max("days_in_shop").alias("max_days"),
        F.round(F.stddev("days_in_shop"), 2).alias("stddev_days")
    )
    .orderBy("store_name", F.desc("avg_days_in_shop"))
)

print("\n⏱️ KPI 2: Average Days in Shop by Store and Service Type")
print("="*80)
display(avg_days_in_shop)

In [0]:
# Survey Coverage
# Number of surveys sent vs responded for each store

survey_coverage = spark.sql(f"""
    SELECT 
        sr.store_id,
        sr.store_name,
        sr.city,
        sr.state,
        COUNT(*) as total_surveys_sent,
        SUM(CASE WHEN sr.responded_flag THEN 1 ELSE 0 END) as total_responses,
        COUNT(*) - SUM(CASE WHEN sr.responded_flag THEN 1 ELSE 0 END) as surveys_pending,
        ROUND(SUM(CASE WHEN sr.responded_flag THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as response_rate_pct,
        RANK() OVER (ORDER BY SUM(CASE WHEN sr.responded_flag THEN 1 ELSE 0 END) * 100.0 / COUNT(*) DESC) as response_rate_rank
    FROM {catalog}.{schema}.cube_survey_detail sr
    GROUP BY sr.store_id, sr.store_name, sr.city, sr.state
    ORDER BY response_rate_pct DESC
""")

print("\n📋 KPI 3: Survey Coverage by Store")
print("="*80)
display(survey_coverage)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
survey_df = spark.table(f"{catalog}.{schema}.cube_survey_detail")
# Filter only responded surveys
survey_clean = (
    survey_df
    .filter(F.col("responded_flag") == True)
    .filter(F.col("overall_satisfaction_rating").isNotNull())
)
# Aggregate scores per store
store_scores = (
    survey_clean
    .groupBy("store_id", "store_name", "manager_name")
    .agg(
        F.round(F.avg("delivered_on_time_rating"), 2).alias("avg_on_time_rating"),
        F.round(F.avg("work_quality_rating"), 2).alias("avg_quality_rating"),
        F.round(F.avg("cleanliness_rating"), 2).alias("avg_cleanliness_rating"),
        F.round(F.avg("communication_rating"), 2).alias("avg_communication_rating"),
        F.round(F.avg("overall_satisfaction_rating"), 2).alias("avg_overall_satisfaction"),
        F.sum(F.when(F.col("overall_satisfaction_rating") >= 8, 1).otherwise(0)).alias("high_satisfaction_count"),
        F.count("*").alias("responded_count")
    )
)

# Compute composite metrics
store_scores = (
    store_scores
    .withColumn(
        "composite_score",
        F.round(
            (
                F.col("avg_on_time_rating") +
                F.col("avg_quality_rating") +
                F.col("avg_cleanliness_rating") +
                F.col("avg_communication_rating") +
                F.col("avg_overall_satisfaction")
            ) / 5,
            2
        )
    )
    .withColumn(
        "high_satisfaction_pct",
        F.round(
            (F.col("high_satisfaction_count") * 100.0 / F.col("responded_count")),
            2
        )
    )
)
# Ranking using window function
window_spec = Window.orderBy(F.col("avg_overall_satisfaction").desc())

survey_final = (
    store_scores
    .withColumn("satisfaction_rank", F.rank().over(window_spec))
    .orderBy("satisfaction_rank")
)
print("\n😊 KPI 4: Survey Scores Summary and Store Rankings")
print("="*80)
display(survey_final)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
finance = spark.table(f"{catalog}.{schema}.cube_financial_detail")
# Aggregate revenue and budget
finance_agg = (
    finance
    .groupBy("period_month", "manager_name", "store_id", "store_name")
    .agg(
        F.round(F.sum("invoice_amount"), 2).alias("actual_revenue"),
        
        # If budget_amount repeats per row, MAX avoids inflation
        F.round(F.max("budget_amount"), 2).alias("budget_amount")
    )
    .withColumn(
        "budget_variance_pct",
        F.round(
            ((F.col("actual_revenue") - F.col("budget_amount")) /
             F.col("budget_amount")) * 100, 2
        )
    )
    .withColumn(
        "budget_status",
        F.when(F.col("budget_variance_pct") >= 0, "Met or Exceeded")
         .otherwise("Below Budget")
    )
)
# Ranking by variance per month
window_spec = Window.partitionBy("period_month").orderBy(F.col("budget_variance_pct").desc())

revenue_vs_budget_final = (
    finance_agg
    .withColumn("manager_rank_by_budget", F.rank().over(window_spec))
    .orderBy(F.col("period_month").desc(), F.col("manager_rank_by_budget"))
)
print("\n💰 KPI 5: Revenue vs Budget by Manager")
print("="*80)
display(revenue_vs_budget_final)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ops = spark.table(f"{catalog}.{schema}.cube_operations_detail")

# Clean data
ops_clean = (
    ops
    .filter(F.col("order_status") == "COMPLETED")
    .filter(F.col("technician_id").isNotNull())
    .filter(F.col("technician_name").isNotNull())
    .filter(F.col("delivery_variance_days").isNotNull())
)

# Technician performance aggregation
tech_perf = (
    ops_clean
    .groupBy("technician_id", "technician_name", "store_id", "store_name")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("delivery_variance_days") <= 0, 1).otherwise(0)).alias("on_time_deliveries"),
        F.round(
            (F.sum(F.when(F.col("delivery_variance_days") <= 0, 1).otherwise(0)) /
             F.count("*")) * 100,
            2
        ).alias("on_time_percentage"),
        
        # Lower is better (negative = early, positive = late)
        F.round(F.avg("delivery_variance_days"), 2).alias("avg_delivery_variance_days")
    )
    .filter(F.col("total_orders") >= 10)   # Minimum threshold
)

# Ranking criteria
window_spec = Window.orderBy(
    F.col("on_time_percentage").desc(),
    F.col("avg_delivery_variance_days").asc()
)

top_technicians_final = (
    tech_perf
    .withColumn("accuracy_rank", F.rank().over(window_spec))
    .orderBy("accuracy_rank")
    .limit(20)
)

print("\n🏆 KPI 6: Top 20 Technicians by Completion Time Accuracy")
print("=" * 80)
display(top_technicians_final)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

finance = spark.table(f"{catalog}.{schema}.cube_financial_detail")

# Determine current year and YTD cutoff date
today = date.today()
current_year = today.year
ytd_cutoff = today  # inclusive
# Current YTD (Jan 1 → today)
current_ytd = (
    finance
    .filter(F.col("period_year") == current_year)
    .filter(F.col("period_month_num") <= today.month)
    .groupBy("store_id", "store_name", "manager_name")
    .agg(
        F.sum("invoice_amount").alias("current_ytd_revenue"),
        F.countDistinct("order_id").alias("current_ytd_orders")
    )
)
# Previous YTD (Jan 1 last year → same month/day last year)
previous_year = current_year - 1

previous_ytd = (
    finance
    .filter(F.col("period_year") == previous_year)
    .filter(F.col("period_month_num") <= today.month)
    .groupBy("store_id")
    .agg(
        F.sum("invoice_amount").alias("prev_ytd_revenue"),
        F.countDistinct("order_id").alias("prev_ytd_orders")
    )
)

# 4. Join current and previous YTD
ytd_joined = (
    current_ytd
    .join(previous_ytd, "store_id", "left")
    .withColumn("prev_ytd_revenue", F.coalesce(F.col("prev_ytd_revenue"), F.lit(0.0)))
    .withColumn("prev_ytd_orders", F.coalesce(F.col("prev_ytd_orders"), F.lit(0)))
)

# Compute KPI metrics
ytd_final = (
    ytd_joined
    .withColumn("ytd_revenue_change",
        F.round(F.col("current_ytd_revenue") - F.col("prev_ytd_revenue"), 2)
    )
    .withColumn("ytd_growth_pct",
        F.round(
            (F.col("current_ytd_revenue") - F.col("prev_ytd_revenue")) /
            F.when(F.col("prev_ytd_revenue") > 0, F.col("prev_ytd_revenue"))
            * 100, 2
        )
    )
)

# Ranking
window_spec = Window.orderBy(F.col("ytd_growth_pct").desc())

ytd_ranked = (
    ytd_final
    .withColumn("growth_rank", F.rank().over(window_spec))
    .orderBy("growth_rank")
)

print("\n📈 KPI 7: Year-to-Date Revenue Growth (Corrected)")
print("=" * 80)
display(ytd_ranked)

In [0]:
from pyspark.sql import functions as F

ops = spark.table(f"{catalog}.{schema}.cube_operations_detail")

# Clean & validate data
ops_clean = (
    ops
    .filter(F.col("days_in_shop").isNotNull())
    .filter(F.col("days_in_shop") >= 0)
    .filter(F.col("days_to_work_start").isNotNull())
    .filter(F.col("days_to_work_start") >= 0)
    .filter(F.col("work_duration_days").isNotNull())
    .filter(F.col("work_duration_days") >= 0)
)

stage_cycle_time = (
    ops_clean
    .groupBy("store_id", "store_name", "service_type")
    .agg(
        F.round(F.avg("days_to_work_start"), 2).alias("avg_days_vehicle_to_work_start"),
        F.round(F.avg("work_duration_days"), 2).alias("avg_days_work_to_completion"),
        F.round(F.avg("days_in_shop"), 2).alias("avg_total_days_in_shop"),
        F.count("*").alias("total_orders")
    )
    .orderBy(F.col("avg_total_days_in_shop").desc())
)

print("\n⏳ KPI 8: Stage-wise Day Cycle Time")
print("=" * 80)
display(stage_cycle_time)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

finance = spark.table(f"{catalog}.{schema}.cube_financial_detail")
#  Clean Data - INITIAL ESTIMATES ONLY
finance_clean = (
    finance
    .filter(F.col("estimator_id").isNotNull())
    .filter(F.col("estimator_name").isNotNull())
    .filter(F.col("is_initial_estimate") == True)  # ← Only initial estimates
    .filter(F.col("actual_amount").isNotNull() & (F.col("actual_amount") > 0))
    .filter(F.col("final_estimate_amount").isNotNull())
)

#  Calculate Variance Explicitly
finance_clean = (
    finance_clean
    .withColumn("estimate_variance", F.col("actual_amount") - F.col("final_estimate_amount"))
    .withColumn(
        "variance_pct",
        F.abs(F.col("estimate_variance")) * 100 / F.col("actual_amount")
    )
)

#  Aggregate Metrics per Estimator
est_metrics = (
    finance_clean
    .groupBy("estimator_id", "estimator_name")
    .agg(
        F.count("*").alias("total_initial_estimates"),

        F.round(F.avg("final_estimate_amount"), 2).alias("avg_initial_estimate"),
        F.round(F.avg("actual_amount"), 2).alias("avg_actual_amount"),
        
        F.round(F.avg("estimate_variance"), 2).alias("avg_variance"),
        F.round(F.avg("variance_pct"), 2).alias("avg_variance_pct"),

        F.sum(F.when(F.col("variance_pct") <= 10, 1).otherwise(0)).alias("estimates_within_10pct")
    )
    .filter(F.col("total_initial_estimates") >= 10)   # Minimum sample threshold
)

#  Scoring Model
est_metrics = (
    est_metrics
    .withColumn(
        "avg_accuracy_score",
        F.round(
            F.when(F.col("avg_variance_pct") <= 5, 100)
            .when(F.col("avg_variance_pct") <= 10, 90)
            .when(F.col("avg_variance_pct") <= 15, 80)
            .when(F.col("avg_variance_pct") <= 20, 70)
            .otherwise(F.greatest(F.lit(0), 100 - F.col("avg_variance_pct"))),
            2
        )
    )
    .withColumn(
        "accuracy_rate_pct",
        F.round((F.col("estimates_within_10pct") * 100.0) / F.col("total_initial_estimates"), 2)
    )
)

#  Ranking by Accuracy Score
window_spec = Window.orderBy(F.col("avg_accuracy_score").desc())

estimator_accuracy_final = (
    est_metrics
    .withColumn("accuracy_rank", F.rank().over(window_spec))
    .orderBy("accuracy_rank")
)

print("\n🎯 KPI 9: Initial Estimator Accuracy Rankings")
print("=" * 80)
display(estimator_accuracy_final)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load tables
fact_orders = spark.table(f"{catalog}.{schema}.fact_orders")
dim_store = spark.table(f"{catalog}.{schema}.dim_store")
dim_technician = spark.table(f"{catalog}.{schema}.dim_technician")

# Clean Data & Join Dimension
orders_clean = (
    fact_orders
    .filter(F.col("days_in_shop").isNotNull())
    .filter(F.col("days_in_shop") >= 0)
    .filter(F.col("days_in_shop") <= 365)  # safety filter
    .filter(F.col("technician_id").isNotNull())
    .filter(F.col("store_id").isNotNull())
)

# Join store dimension
orders_with_store = (
    orders_clean
    .join(dim_store.select("store_id", "store_name"), "store_id", "left")
)

# Join technician dimension only on technician_id
orders_with_metrics = (
    orders_with_store
    .join(dim_technician.select("technician_id", "technician_name"), "technician_id", "left")
    .withColumn(
        "work_month",
        F.date_format(F.col("vehicle_in_datetime"), "yyyy-MM")
    )
)

# Aggregate Workload
technician_workload = (
    orders_with_metrics
    .groupBy("work_month", "store_id", "store_name", "technician_id", "technician_name")
    .agg(
        F.count("*").alias("orders_handled"),
        F.round(F.sum("days_in_shop"), 2).alias("total_days_in_shop"),
        F.round(F.avg("days_in_shop"), 2).alias("avg_days_per_order")
    )
)

# Ranking (Top 20 per Month)
window_spec = Window.partitionBy("work_month").orderBy(
    F.desc("orders_handled"),
    F.desc("total_days_in_shop"),
    F.asc("avg_days_per_order")
)

technician_workload_ranked = (
    technician_workload
    .withColumn("workload_rank", F.rank().over(window_spec))
)

# Display only Top 20 Technicians Per Month
print("\n👷 KPI 10: Technician Workload Analysis by Month (Corrected)")
print("=" * 80)
display(technician_workload_ranked.filter(F.col("workload_rank") <= 20))